<a href="https://colab.research.google.com/github/BF667-IDLE/Hyper-RVC/blob/main/assets/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hypr RVC**


- made by [BF667](https://github.com/BF667-IDLE)

- Improved the Start UI Cell with other Tunnel types by [Nick088](https://linktr.ee/Nick088)


<br>This colab uses the following projects:
- [Music Source Separation Universal Training Code](https://github.com/ZFTurbo/Music-Source-Separation-Training) by [ZFTurbo](https://github.com/ZFTurbo)
- [Applio](https://github.com/IAHispano/Applio) by [IAHispano](https://github.com/IAHispano)

In [ ]:
#@title ## **Install**


import os
import codecs
from IPython.display import clear_output
print("Installing requirements")
repo = "https://github.com/BF667-IDLE/Hyper-RVC.git"
!git clone $repo main_program  &> /dev/null
%cd main_program
!apt-get install -y portaudio19-dev -qq
!pip install uv pyngrok -q
!uv pip install --no-deps -r requirements.txt -q
!python main/utils.py
clear_output()
print("Requirements installed!")

In [ ]:
#@title ## **Start UI**

#@markdown The type of tunnel you wanna use for seeing the public link, so that if one of them is down, you can use the other one.
Tunnel = "Gradio" #@param ["Gradio", "Ngrok", "Cloudflare"]

#@markdown When using Ngrok or Cloudflare, wait for the Local URL to appear, then use the Public URL shown above.

#@markdown Use the following option **only if you chose Ngrok** as the Tunnel:

#@markdown You can get the Ngrok Authtoken here: https://dashboard.ngrok.com/tunnels/authtokens/new.

ngrok_authtoken = "" #@param {type:"string"}

# @markdown You can optionally change the Ngrok Tunnel Region to one nearer to you for lower latency

ngrok_region = "us - United States (Ohio)" # @param ["au - Australia (Sydney)","eu - Europe (Frankfurt)", "ap - Asia/Pacific (Singapore)", "us - United States (Ohio)", "jp - Japan (Tokyo)", "in - India (Mumbai)","sa - South America (Sao Paulo)"]


import codecs
from IPython.display import clear_output

RUNTIME = codecs.decode("znva.cl", "rot_13")

if Tunnel == "Gradio":
    share_flag = "--share"
elif Tunnel == "Ngrok":
  share_flag = ""
  from pyngrok import conf, ngrok
  NgrokConfig = conf.PyngrokConfig()
  NgrokConfig.auth_token = ngrok_authtoken
  NgrokConfig.region = ngrok_region[0:2]
  conf.set_default(NgrokConfig)
  main_tunnel = ngrok.connect(7755)
  print("Ngrok Tunnel Public URL:", main_tunnel.public_url)
elif Tunnel == "Cloudflare":
  share_flag = ""
  # install cloudflared
  !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
  !chmod +x /usr/local/bin/cloudflared
  import time
  # run cloudflared
  with open('url.txt', 'w') as file:
        file.write('')

  get_ipython().system_raw('cloudflared tunnel --url http://localhost:7755 >> url.txt 2>&1 &')

  time.sleep(4)

  with open('url.txt', 'r') as file:
      tunnel_url = !grep -oE "https://[a-zA-Z0-9.-]+\.trycloudflare\.com" url.txt
      tunnel_url = tunnel_url[0]

  clear_output()

  print(f"Cloudflare Tunnel Public URL: \033[0m\033[93m{tunnel_url}\033[0m")

# kills previously running processes
!fuser -k 7755/tcp

command = f"python {RUNTIME} {share_flag}"
!{command}